In [2]:
import os
import numpy as np
import librosa
from glob import glob
from sklearn.metrics import roc_auc_score
import tensorflow as tf
from tensorflow.keras import layers, models

In [3]:
DATA_DIR = "processed_data"
SR = 16000
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512

WINDOW_FRAMES = 256
WINDOW_HOP = 128

BATCH_SIZE = 16
EPOCHS = 30

In [4]:
def extract_log_mel(y):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )
    return librosa.power_to_db(mel)


def sliding_windows(mel):
    windows = []
    for start in range(0, mel.shape[1] - WINDOW_FRAMES + 1, WINDOW_HOP):
        window = mel[:, start:start + WINDOW_FRAMES]
        windows.append(window)
    return windows

In [5]:
def load_train_data():
    files = glob(os.path.join(DATA_DIR, "train", "normal", "*.wav"))
    X = []

    for f in files:
        y, _ = librosa.load(f, sr=SR)
        mel = extract_log_mel(y)

        # Use ONLY first window for training (standard baseline)
        if mel.shape[1] >= WINDOW_FRAMES:
            mel = mel[:, :WINDOW_FRAMES]
        else:
            mel = np.pad(mel, ((0, 0), (0, WINDOW_FRAMES - mel.shape[1])))

        X.append(mel)

    X = np.array(X)[..., np.newaxis]
    return X

In [6]:
def build_autoencoder():
    model = models.Sequential([
        layers.Input(shape=(N_MELS, WINDOW_FRAMES, 1)),

        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(2),

        layers.Conv2D(16, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(2),

        layers.Conv2D(8, 3, activation="relu", padding="same"),

        layers.UpSampling2D(2),
        layers.Conv2D(16, 3, activation="relu", padding="same"),

        layers.UpSampling2D(2),
        layers.Conv2D(1, 3, activation="linear", padding="same")
    ])

    model.compile(optimizer="adam", loss="mse")
    return model

In [7]:
print("Loading training data...")
X_train = load_train_data()

print("Training model...")
model = build_autoencoder()
model.fit(
    X_train,
    X_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True
)

Loading training data...
Training model...
Epoch 1/30
65/65 ━━━━━━━━━━━━━━━━━━━━ 6s 87ms/step - loss: 300.7045
Epoch 2/30
65/65 ━━━━━━━━━━━━━━━━━━━━ 5s 83ms/step - loss: 18.4759
Epoch 3/30
65/65 ━━━━━━━━━━━━━━━━━━━━ 6s 91ms/step - loss: 12.5000
Epoch 4/30
65/65 ━━━━━━━━━━━━━━━━━━━━ 6s 91ms/step - loss: 10.9821
Epoch 5/30
65/65 ━━━━━━━━━━━━━━━━━━━━ 6s 91ms/step - loss: 10.3257
Epoch 6/30
65/65 ━━━━━━━━━━━━━━━━━━━━ 6s 94ms/step - loss: 9.8694
Epoch 7/30
65/65 ━━━━━━━━━━━━━━━━━━━━ 6s 99ms/step - loss: 9.5225 
Epoch 8/30
65/65 ━━━━━━━━━━━━━━━━━━━━ 6s 97ms/step - loss: 9.2594
Epoch 9/30
65/65 ━━━━━━━━━━━━━━━━━━━━ 7s 101ms/step - loss: 9.0731
Epoch 10/30
65/65 ━━━━━━━━━━━━━━━━━━━━ 6s 96ms/step - loss: 8.9469
Epoch 11/30
65/65 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - loss: 8.8602
Epoch 12/30
65/65 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - loss: 8.7643
Epoch 13/30
65/65 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - loss: 8.6967
Epoch 14/30
65/65 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - loss: 8.6431
Epoch 15/30
65/65 ━━

In [11]:
def anomaly_score_file(path):
    y, _ = librosa.load(path, sr=SR)
    mel = extract_log_mel(y)

    if mel.shape[1] < WINDOW_FRAMES:
        mel = np.pad(mel, ((0, 0), (0, WINDOW_FRAMES - mel.shape[1])))

    windows = sliding_windows(mel)

    scores = []
    for w in windows:
        w = w[np.newaxis, ..., np.newaxis]
        recon = model.predict(w, verbose=0)
        err = np.mean((w - recon) ** 2)
        scores.append(err)

    return np.mean(scores)

def evaluate(split, label):
    files = glob(os.path.join(DATA_DIR, split, label, "*.wav"))
    return [anomaly_score_file(f) for f in files]

In [12]:
print("Evaluating...")
scores_normal = evaluate("test", "normal")
scores_abnormal = evaluate("test", "abnormal")

y_true = np.concatenate([
    np.zeros(len(scores_normal)),
    np.ones(len(scores_abnormal))
])

scores = np.concatenate([scores_normal, scores_abnormal])

auc = roc_auc_score(y_true, scores)
print(f"\nROC-AUC Score (Sliding Window): {auc:.4f}")

Evaluating...

ROC-AUC Score (Sliding Window): 0.6103
